In [21]:
import requests
import numpy as np
import hdbscan
import matplotlib.pyplot as plt
from sklearn.preprocessing import normalize
import umap
import random
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import HyperbandPruner
import pandas as pd
from sklearn.metrics import f1_score, fbeta_score
import joblib
from sentence_transformers import SentenceTransformer
from sklearn.utils import resample

In [15]:
cities = {
    "Москва": (55.75, 37.62),
    "Санкт-Петербург": (59.93, 30.32),
    "Новосибирск": (54.99, 82.90),
    "Екатеринбург": (56.83, 60.60),
    "Казань": (55.78, 49.12),
    "Челябинск": (55.15, 61.43),
    "Омск": (54.99, 73.37),
    "Самара": (53.20, 50.15),
    "Ростов-на-Дону": (47.23, 39.72),
    "Уфа": (54.73, 55.94),
    "Красноярск": (56.01, 92.87),
    "Пермь": (58.01, 56.23),
    "Воронеж": (51.67, 39.18),
    "Волгоград": (48.71, 44.51),
    "Краснодар": (45.04, 38.98),
    "Иркутск": (52.29, 104.28),
    "Хабаровск": (48.48, 135.08),
    "Владивосток": (43.11, 131.88),
    "Тюмень": (57.15, 68.99),
    "Барнаул": (53.35, 83.75),
}

weather_descriptions = {
    0: "ясно", 1: "преимущественно ясно", 2: "переменная облачность",
    3: "пасмурно", 45: "туман", 48: "изморозь", 51: "лёгкая морось",
    53: "умеренная морось", 61: "небольшой дождь", 63: "умеренный дождь",
    65: "сильный дождь", 71: "небольшой снег", 73: "умеренный снег",
    75: "сильный снег", 80: "ливень", 95: "гроза",
}


def clean_text(text):
    return text.replace('\xa0', ' ').strip()


def fetch_weather_texts(cities):
    texts = []
    for city, (lat, lon) in cities.items():
        url = "https://api.open-meteo.com/v1/forecast"
        params = {
            "latitude": lat, "longitude": lon,
            "hourly": "temperature_2m,windspeed_10m,weathercode",
            "forecast_days": 16,
            "past_days": 30
        }
        r = requests.get(url, params=params)
        data = r.json()
        hourly = data["hourly"]

        for i in range(0, len(hourly["time"]), 3):
            code = hourly["weathercode"][i]
            desc = weather_descriptions.get(code, "облачно")
            temp = hourly["temperature_2m"][i]
            wind = hourly["windspeed_10m"][i]
            if temp is None or wind is None or code is None:
                continue
            text = f"В городе {city}: {desc}, температура {temp:.0f}°C, ветер {wind:.0f} км/ч."
            texts.append(text)

    return texts



In [16]:
import pandas as pd

df = pd.read_csv("lenta-ru-news.csv")
# Аномальные — происшествия, катастрофы, политика
anomaly_df = df[df["title"].str.contains("происшествия|катастроф|чрезвычайн|война|конфликт|авария",
                                          case=False, na=False)]
anomaly_test_texts = anomaly_df["title"].dropna().sample(frac=1, random_state=42).tolist()[:50]
anomaly_val_texts = anomaly_df["title"].dropna().sample(frac=1, random_state=42).tolist()[50:200]

anomaly_test_texts = [clean_text(t) for t in anomaly_test_texts]
anomaly_val_texts = [clean_text(t) for t in anomaly_val_texts]
print(f"Тестовых Аномалий: {len(anomaly_test_texts)}")





C:\Users\Админ\AppData\Local\Temp\ipykernel_2828\1936340579.py:3: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("lenta-ru-news.csv")


Тестовых Аномалий: 50


In [17]:
all_weather_texts = fetch_weather_texts(cities)
all_weather_texts = [clean_text(t) for t in all_weather_texts]


random.seed(42)
random.shuffle(all_weather_texts)

print(f"Всего погодных: {len(all_weather_texts)}")

# 40% train, 40% val, 20% test
split = int(len(all_weather_texts) * 0.4)
val_split = int(len(all_weather_texts) * 0.8)

train_weather_texts = all_weather_texts[:split]
val_weather_texts = all_weather_texts[split:val_split]
test_weather_texts = all_weather_texts[val_split:]


print(f"Train: {len(train_weather_texts)}, Val: {len(val_weather_texts)}, Test: {len(test_weather_texts)}")

# Финальный датасет
train_texts = train_weather_texts
val_texts = val_weather_texts + anomaly_val_texts
test_texts = test_weather_texts + anomaly_test_texts

test_labels = [0]*len(test_weather_texts) + [1]*len(anomaly_test_texts)
val_labels = [0]*len(val_weather_texts) + [1]*len(anomaly_val_texts)

Всего погодных: 7360
Train: 2944, Val: 2944, Test: 1472


Векторизация

In [18]:
print(f"\nЗагрузка моделей")
model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
model2 = SentenceTransformer("all-MiniLM-L6-v2")
model3 = SentenceTransformer("cointegrated/rubert-tiny2")
model4 = SentenceTransformer("paraphrase-multilingual-mpnet-base-v2")



Загрузка моделей


In [19]:
pipeline = joblib.load("pipeline_hdbscan_v1.1.pkl")
models = [SentenceTransformer(n) for n in pipeline["encoders"]]

embs = []
for model, reducer in zip(models, pipeline["reducers"]):
    emb = model.encode(test_texts, show_progress_bar = True, batch_size = 32)
    emb = normalize(emb, norm='l2')
    embs.append(reducer.transform(emb)) 
    
x_test = np.concatenate(embs, axis=1)




Batches:   0%|          | 0/48 [00:00<?, ?it/s]

Batches:   0%|          | 0/48 [00:00<?, ?it/s]

Batches:   0%|          | 0/48 [00:00<?, ?it/s]

Оценка модели на тестовой выборке



на валидационной выборке был найден оптимальный порог силы - 0.075 во время тюнинга модели

In [20]:
predicted_cluster_labels, test_strengths = hdbscan.approximate_predict(pipeline["clusterer"], x_test)
predictions = (test_strengths < pipeline["threshold"]).astype(int)

test_labels_arr = np.array(test_labels)  # ИСТИННЫЕ метки (0=погода, 1=аномалия)

tp = ((predictions == 1) & (test_labels_arr == 1)).sum()
fp = ((predictions == 1) & (test_labels_arr == 0)).sum()
fn = ((predictions == 0) & (test_labels_arr == 1)).sum()
tn = ((predictions == 0) & (test_labels_arr == 0)).sum()

n_anom = len(anomaly_test_texts)
recall = tp / n_anom if n_anom > 0 else 0
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print(f"TP: {tp}/{n_anom}")
print(f"FP: {fp}")
print(f"FN: {fn}")
print(f"TN: {tn}")
print(f"Recall:    {recall:.2%}")
print(f"Precision: {precision:.2%}")
print(f"F1:        {f1:.4f}")

TP: 47/50
FP: 172
FN: 3
TN: 1300
Recall:    94.00%
Precision: 21.46%
F1:        0.3494


In [ ]:
n_bootstrap = 1000

f1_scores = []
recall_scores = []
precision_scores = []

test_labels_arr = np.array(test_labels)


for _ in range(n_bootstrap):
    # Случайно выбираем точки с возвращением
    idx = resample(np.arange(len(test_labels_arr)), replace=True)
    y_true_sample = test_labels_arr[idx]
    y_pred_sample = predictions[idx]
    
    f1_scores.append(f1_score(y_true_sample, y_pred_sample, zero_division=0))
    
    tp = ((y_pred_sample == 1) & (y_true_sample == 1)).sum()
    fp = ((y_pred_sample == 1) & (y_true_sample == 0)).sum()
    fn = ((y_pred_sample == 0) & (y_true_sample == 1)).sum()
    
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall_scores.append(rec)
    precision_scores.append(prec)


# 95% ДИ — берём 2.5 и 97.5 перцентили
print(f"F1:        {np.mean(f1_scores):.3f} [{np.percentile(f1_scores, 2.5):.3f}, {np.percentile(f1_scores, 97.5):.3f}]")
print(f"Recall:    {np.mean(recall_scores):.3f} [{np.percentile(recall_scores, 2.5):.3f}, {np.percentile(recall_scores, 97.5):.3f}]")
print(f"Precision: {np.mean(precision_scores):.3f} [{np.percentile(precision_scores, 2.5):.3f}, {np.percentile(precision_scores, 97.5):.3f}]")

F1:        0.348 [0.273, 0.419]
Recall:    0.941 [0.867, 1.000]
Precision: 0.214 [0.161, 0.268]
